# AI Historical Shorts — Free Google Colab GPU

This notebook runs the repository's Wan 2.1 1.3B video pipeline on a Colab GPU, creates Hindi narration and subtitles, and sends the finished MP4 to Telegram.

## Before running
1. Open **Runtime → Change runtime type → T4 GPU** (if available).
2. In the left **Secrets** panel, add `GEMINI_API_KEY`, `TELEGRAM_BOT_TOKEN`, and `TELEGRAM_CHAT_ID`. Enable notebook access for each secret.
3. Run the cells from top to bottom. GPU availability and session duration are controlled by Colab; free access is not guaranteed.

The notebook does not use Kaggle. It clones the `wan-kaggle-test` branch containing the existing generator.

In [ ]:
# Check that Colab assigned a GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU assigned. In Colab choose Runtime → Change runtime type → T4 GPU, reconnect, and run again.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GB')

In [ ]:
# Install the generator dependencies and FFmpeg
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install --upgrade diffusers==0.43.0 accelerate transformers ftfy sentencepiece edge-tts requests google-genai

In [ ]:
# Clone the existing video-generation code
%cd /content
!rm -rf Shorts-pipeline
!git clone --depth 1 --branch wan-kaggle-test https://github.com/echo-kernel/Shorts-pipeline.git Shorts-pipeline
%cd /content/Shorts-pipeline
import os, sys, json, time
print('Repository ready:', os.getcwd())

In [ ]:
# Read private keys from Colab Secrets (never paste keys into notebook code)
from google.colab import userdata
required = ['GEMINI_API_KEY', 'TELEGRAM_BOT_TOKEN', 'TELEGRAM_CHAT_ID']
missing = []
for key in required:
    try:
        value = userdata.get(key)
        if not value:
            missing.append(key)
        else:
            os.environ[key] = value.strip()
    except Exception:
        missing.append(key)
if missing:
    raise RuntimeError('Add these names in Colab Secrets and enable notebook access: ' + ', '.join(sorted(set(missing))))
print('All required secrets are available. Values are hidden.')

In [ ]:
# Choose the historical topic and generate the story with Gemini
from google import genai
from getpass import getpass
topic = input('Historical topic [U Tirot Sing and the Khasi resistance in 1829]: ').strip()
if not topic:
    topic = 'U Tirot Sing and the Khasi resistance against British expansion in 1829'
prompt = f'''Create a historically responsible YouTube Short about: {topic}
Return ONLY valid JSON with this exact structure:
{{\"title\":\"Short engaging title\",\"voiceover\":\"Natural Hindi documentary narration, approximately 105 to 120 words\",\"scenes\":[\"Cinematic English video prompt 1\",\"Cinematic English video prompt 2\",\"Cinematic English video prompt 3\",\"Cinematic English video prompt 4\"]}}
Requirements: historically accurate; no invented quotations/events; four connected cinematic 3D animated historical scenes; consistent character appearance and art direction; grounded clothing, architecture and props; natural character movement; dramatic but historically plausible lighting; vertical 9:16 framing; no modern objects, text, logos or watermarks. Hindi narration and English scene prompts. Return JSON only.'''
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
story = None
errors = []
for model in ['gemini-2.5-flash', 'gemini-2.5-flash-lite']:
    try:
        print('Trying story model:', model)
        response = client.models.generate_content(model=model, contents=prompt, config={'response_mime_type':'application/json'})
        candidate = json.loads(response.text.strip())
        assert candidate.get('title') and candidate.get('voiceover')
        assert isinstance(candidate.get('scenes'), list) and len(candidate['scenes']) == 4
        assert all(isinstance(s, str) and s.strip() for s in candidate['scenes'])
        candidate['gemini_model'] = model
        story = candidate
        break
    except Exception as e:
        errors.append(f'{model}: {type(e).__name__}: {e}')
if story is None:
    raise RuntimeError('Gemini story generation failed:\n' + '\n'.join(errors))
os.environ['VIDEO_STORY_JSON'] = json.dumps(story, ensure_ascii=False)
os.environ['VIDEO_TOPIC'] = topic
print('\nTitle:', story['title'])
print('Model:', story['gemini_model'])
print('Story ready; API key is not printed.')

In [ ]:
# Generate the four AI scenes, Hindi narration, subtitles and final MP4.
# The existing pipeline also sends the completed MP4 and progress updates to Telegram.
import generate
generate.main()

from pathlib import Path
video = Path('/content/Shorts-pipeline/output/historical_short.mp4')
if not video.exists() or video.stat().st_size < 10000:
    raise RuntimeError('Final MP4 was not created or is unexpectedly small.')
print(f'\nVerified MP4: {video} ({video.stat().st_size/1024/1024:.1f} MB)')

In [ ]:
# Optional: download a local copy to your phone
from google.colab import files
files.download('/content/Shorts-pipeline/output/historical_short.mp4')